In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import statsmodels.api as sm

from ydata_profiling import ProfileReport
import pygwalker as pyg


## EDA

In [17]:
df = pd.read_excel('data/illegal-entry-routes-to-the-uk-dataset-dec-2025.xlsx', sheet_name='Data_IER_D01')
profile = ProfileReport(df, title="YData Profiling Report")

In [18]:
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 1103.58it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
walker = pyg.walk(df)

Box(children=(HTML(value='\n<div id="ifr-pyg-00064f694ee9a4101ND2drki3Cj5GcIA" style="height: auto">\n    <hea…

# Modelling approach

## Mixed Effects Model — Country Prioritisation

### Why mixed effects?

We have a **panel** of 111 nationalities observed over 32 quarters (2018 Q1 – 2025 Q4). Nationalities vary hugely in volume, and they nest naturally within world regions. A mixed effects model handles this by:

- **Fixed effects** capture the overall relationship between arrivals and predictors (time trend, lagged returns).
- **Random intercepts** by nationality let each country have its own baseline level — essentially partial pooling between a fully-pooled OLS and fully-separate per-country regressions. Smaller countries borrow strength from the population mean.
- **Random slopes** on lagged returns let the *effectiveness* of returns vary by country: for some nationalities returns may be associated with reduced future arrivals; for others, not.

### Specification

$$\log(1 + \text{arrivals}_{c,t}) = \underbrace{(\beta_0 + u_{0c})}_{\text{intercept}} + \beta_1 \cdot t + \underbrace{(\beta_2 + u_{2c})}_{\text{returns effect}} \cdot \log(1 + \text{returns}_{c,t-1}) + \varepsilon_{c,t}$$

Where $u_{0c}$ is the random intercept and $u_{2c}$ the random slope for nationality $c$, both drawn from a bivariate normal.

### Caveat: returns are endogenous

Returns are a **consequence** of arrivals, not an independent intervention. 

In [23]:
# ── Build nationality × quarter panel ────────────────────────────────────────

df_arrivals = pd.read_excel(
    'data/illegal-entry-routes-to-the-uk-dataset-dec-2025.xlsx',
    sheet_name='Data_IER_D01', header=1,
)
df_returns = pd.read_excel(
    'data/returns-datasets-dec-2025.xlsx',
    sheet_name='Data_Ret_D01', header=1,
)

# Small-boat arrivals aggregated to nationality × quarter
df_sb = df_arrivals[df_arrivals['Method of entry'] == 'Small boat arrivals']
arr = (
    df_sb.groupby(['Nationality', 'Region', 'Year', 'Quarter'])['Number of detections']
    .sum().reset_index()
)
arr.columns = ['Nationality', 'Region', 'Year', 'Quarter', 'arrivals']

# Returns aggregated to nationality × quarter
ret = (
    df_returns.groupby(['Nationality', 'Year', 'Quarter'])['Number of returns']
    .sum().reset_index()
)
ret.columns = ['Nationality', 'Year', 'Quarter', 'returns']

# Merge
panel = arr.merge(ret, on=['Nationality', 'Year', 'Quarter'], how='left')
panel['returns'] = panel['returns'].fillna(0)

# Numeric time index (quarters since 2018 Q1)
quarter_order = sorted(panel['Quarter'].unique())
panel['time_idx'] = panel['Quarter'].map({q: i for i, q in enumerate(quarter_order)})

# Lag returns by 1 quarter within each nationality
panel = panel.sort_values(['Nationality', 'time_idx'])
panel['returns_lag1'] = panel.groupby('Nationality')['returns'].shift(1).fillna(0)

# Log-transform (log1p handles zeros)
panel['log_arrivals'] = np.log1p(panel['arrivals'])
panel['log_returns_lag1'] = np.log1p(panel['returns_lag1'])

# Keep nationalities with ≥ 8 quarterly observations for stable estimation
obs_per_nat = panel.groupby('Nationality').size()
panel = panel[panel['Nationality'].isin(obs_per_nat[obs_per_nat >= 8].index)].copy()

print(f"Panel: {len(panel):,} obs  |  {panel['Nationality'].nunique()} nationalities  |  "
      f"{panel['Quarter'].nunique()} quarters  |  {panel['Region'].nunique()} regions")
panel[['arrivals', 'returns', 'returns_lag1', 'time_idx']].describe().round(1)

Panel: 1,006 obs  |  50 nationalities  |  31 quarters  |  10 regions


,arrivals,returns,returns_lag1,time_idx
count,1006.0,1006.0,1006.0,1006.0
mean,191.3,101.3,95.0,19.9
std,508.3,283.0,271.9,7.3
min,1.0,0.0,0.0,0.0
25%,4.0,6.0,4.0,14.0
50%,17.0,23.5,19.0,20.0
75%,132.8,71.0,68.0,26.0
max,9044.0,2658.0,2658.0,31.0


In [24]:
# ── Model A: Random intercepts only ──────────────────────────────────────────
import statsmodels.formula.api as smf

md_ri = smf.mixedlm(
    'log_arrivals ~ time_idx + log_returns_lag1',
    data=panel,
    groups=panel['Nationality'],
)
fit_ri = md_ri.fit(reml=True)
print(fit_ri.summary())

          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: log_arrivals
No. Observations: 1006    Method:             REML        
No. Groups:       50      Scale:              1.0366      
Min. group size:  9       Log-Likelihood:     -1552.5035  
Max. group size:  30      Converged:          Yes         
Mean group size:  20.1                                    
----------------------------------------------------------
                 Coef. Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept        1.211    0.271  4.466 0.000  0.680  1.742
time_idx         0.079    0.005 15.364 0.000  0.069  0.089
log_returns_lag1 0.047    0.037  1.273 0.203 -0.026  0.120
Group Var        3.027    0.632                           



In [25]:
# ── Model B: Random intercepts + random slopes on lagged returns ─────────────

md_rs = smf.mixedlm(
    'log_arrivals ~ time_idx + log_returns_lag1',
    data=panel,
    groups=panel['Nationality'],
    re_formula='~log_returns_lag1',
)
fit_rs = md_rs.fit(reml=True)
print(fit_rs.summary())

                 Mixed Linear Model Regression Results
Model:                 MixedLM     Dependent Variable:     log_arrivals
No. Observations:      1006        Method:                 REML        
No. Groups:            50          Scale:                  0.9991      
Min. group size:       9           Log-Likelihood:         -1536.2684  
Max. group size:       30          Converged:              Yes         
Mean group size:       20.1                                            
-----------------------------------------------------------------------
                             Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------
Intercept                     1.245    0.202  6.151 0.000  0.848  1.641
time_idx                      0.081    0.005 15.836 0.000  0.071  0.091
log_returns_lag1             -0.009    0.051 -0.182 0.856 -0.109  0.091
Group Var                     1.420    0.396                           
Group x l

In [26]:
# ── Extract country-level effects from the random-slopes model ───────────────

re_df = pd.DataFrame({
    nat: vals for nat, vals in fit_rs.random_effects.items()
}).T
re_df.columns = ['re_intercept', 're_returns_slope']

# Total country-specific returns effect = fixed β₂ + random u₂c
beta_returns = fit_rs.fe_params['log_returns_lag1']
re_df['total_returns_effect'] = beta_returns + re_df['re_returns_slope']

# Add total arrivals for context
total_arr = panel.groupby('Nationality')['arrivals'].sum()
re_df = re_df.join(total_arr)

# Map region back
nat_region = panel.drop_duplicates('Nationality').set_index('Nationality')['Region']
re_df = re_df.join(nat_region)

re_df = re_df.sort_values('re_intercept', ascending=False)
re_df[['arrivals', 'Region', 're_intercept', 'total_returns_effect']].head(20).round(3)

,arrivals,Region,re_intercept,total_returns_effect
Iran,30269,Middle East,2.302,0.398
Eritrea,19154,Africa Sub-Saharan,2.000,0.318
Syria,14744,Middle East,1.988,0.319
Sudan,12633,Africa North,1.971,0.292
Afghanistan,27422,Asia Central,1.941,0.280
Iraq,19157,Middle East,1.840,0.307
Not currently recorded,3749,Not currently recorded,1.725,0.260
Vietnam,8429,Asia South East,1.437,0.238
Ethiopia,3639,Africa Sub-Saharan,1.009,0.188
Yemen,3275,Middle East,0.998,0.143


In [27]:
# ── Visualise: country random effects ────────────────────────────────────────

top20 = re_df.head(20).sort_values('re_intercept')

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

# Left panel: random intercept (baseline arrival level)
colours_int = plt.cm.Reds(np.linspace(0.3, 0.9, len(top20)))
axes[0].barh(top20.index, top20['re_intercept'], color=colours_int)
axes[0].set_xlabel('Random Intercept (log-scale baseline)')
axes[0].set_title('Baseline Arrival Level by Nationality\n(higher = more arrivals, controlling for time & returns)')
axes[0].axvline(0, color='grey', linewidth=0.5)

# Right panel: total returns effect (fixed + random slope)
colours_ret = np.where(top20['total_returns_effect'] > 0, '#e74c3c', '#27ae60')
axes[1].barh(top20.index, top20['total_returns_effect'], color=colours_ret)
axes[1].set_xlabel('Total Returns Effect (β₂ + u₂c)')
axes[1].set_title('Returns–Arrivals Association\n(positive = returns NOT reducing future arrivals)')
axes[1].axvline(0, color='grey', linewidth=0.5)

fig.suptitle(
    'Mixed Effects Model — Top 20 Nationalities by Baseline Arrival Level',
    fontsize=14, fontweight='bold', y=1.01,
)
fig.tight_layout()
plt.savefig('presentation/model_mixed_effects.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/55/92g1s_gx6bqf0mxxtwxlddt00000gn/T/ipykernel_10449/3718715585.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
# ── Priority ranking: combine baseline level with returns ineffectiveness ────
#
# Countries that score highest on BOTH dimensions (high baseline arrivals AND
# returns not associated with reductions) are the top intervention priorities.

re_df['intercept_rank'] = re_df['re_intercept'].rank(ascending=False)
re_df['returns_rank'] = re_df['total_returns_effect'].rank(ascending=False)  # high = worse
re_df['combined_rank'] = (re_df['intercept_rank'] + re_df['returns_rank']) / 2

priority = re_df.sort_values('combined_rank').head(15)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    priority['re_intercept'],
    priority['total_returns_effect'],
    s=np.sqrt(priority['arrivals']) * 3,
    c=priority['combined_rank'],
    cmap='RdYlGn',
    edgecolors='black',
    linewidth=0.5,
    alpha=0.85,
)

for nat, row in priority.iterrows():
    ax.annotate(
        nat, (row['re_intercept'], row['total_returns_effect']),
        fontsize=9, ha='left', va='bottom',
        xytext=(5, 5), textcoords='offset points',
    )

ax.set_xlabel('Random Intercept (baseline arrival level)')
ax.set_ylabel('Returns Effect (higher = returns less effective)')
ax.set_title(
    'Country Intervention Priority\n'
    'Top-right = high arrivals AND returns not working → highest priority',
    fontsize=13, fontweight='bold',
)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

plt.colorbar(scatter, label='Combined Rank (lower = higher priority)', shrink=0.7)
fig.tight_layout()
plt.savefig('presentation/model_mixed_priority_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 15 intervention priorities (mixed effects model):")
print(priority[['arrivals', 'Region', 're_intercept', 'total_returns_effect', 'combined_rank']]
      .round(3).to_string())


Top 15 intervention priorities (mixed effects model):
                        arrivals                  Region  re_intercept  total_returns_effect  combined_rank
Iran                       30269             Middle East         2.302                 0.398            1.0
Eritrea                    19154      Africa Sub-Saharan         2.000                 0.318            2.5
Syria                      14744             Middle East         1.988                 0.319            2.5
Sudan                      12633            Africa North         1.971                 0.292            4.5
Iraq                       19157             Middle East         1.840                 0.307            5.0
Afghanistan                27422            Asia Central         1.941                 0.280            5.5
Not currently recorded      3749  Not currently recorded         1.725                 0.260            7.0
Vietnam                     8429         Asia South East         1.437           

/var/folders/55/92g1s_gx6bqf0mxxtwxlddt00000gn/T/ipykernel_10449/3663942716.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model 3: Source–Transit Decomposition

### Motivation

The mixed effects model ranks countries by arrival volume but cannot distinguish **why** a nationality ranks high — is it because the source country is generating a massive outflow across all of Europe, or because the UK is disproportionately targeted? This distinction matters for choosing the right intervention:

- **High Push, low RouteShare** (e.g. Syria): huge outflows to Europe, but the UK gets a small fraction. Source-country intervention reduces the global flow but the UK captures only a small share of that benefit. The main gain is diplomatic (burden-sharing with EU partners).
- **Low Push, high RouteShare** (e.g. Albania): moderate European flow, but the UK gets a disproportionate share. A bilateral deal with the source country (or transit interdiction) directly benefits the UK.

### Decomposition

For each nationality $c$ and year $t$:

$$\text{arrivals\_UK}(c,t) = \underbrace{\text{Push}(c,t)}_{\text{EU27 total detections}} \;\times\; \underbrace{\text{RouteShare}(c,t)}_{\text{UK arrivals} / \text{EU27 detections}}$$

- **Push** comes from Eurostat EU27 data (`migr_eipre`, reason=TOTAL, 2018–2024).
- **RouteShare** is computed as the ratio of UK small-boat arrivals to EU27 total detections for the same nationality and year.

### Notes

- Eurostat data is **annual** (UK data is quarterly, so we aggregate UK arrivals to annual for the join).
- The EU27 figure counts all detected illegal presence (entry + overstay + other), not just small boats. This is a broader measure of "push" but is the best available proxy.
- Post-Brexit, the UK is no longer in EU statistics, so UK arrivals are genuinely additive to the EU27 total — there is no double-counting.

In [29]:
# ── Build the Push × RouteShare panel ────────────────────────────────────────

# Nationality label → Eurostat ISO-2 code
NAT_TO_ISO = {
    'Iran': 'IR', 'Afghanistan': 'AF', 'Iraq': 'IQ', 'Eritrea': 'ER',
    'Albania': 'AL', 'Syria': 'SY', 'Sudan': 'SD', 'Vietnam': 'VN',
    'Turkey': 'TR', 'Somalia': 'SO', 'Ethiopia': 'ET', 'Egypt': 'EG',
    'Yemen': 'YE', 'India': 'IN', 'Kuwait': 'KW', 'Georgia': 'GE',
    'Libya': 'LY', 'Pakistan': 'PK', 'Nigeria': 'NG', 'Sri Lanka': 'LK',
    'South Sudan': 'SS', 'Palestine': 'PS', 'Algeria': 'DZ', 'Morocco': 'MA',
    'Bangladesh': 'BD', 'China': 'CN', 'Cameroon': 'CM', 'Chad': 'TD',
    'Mali': 'ML', 'Niger': 'NE', 'Tunisia': 'TN', 'Ivory Coast': 'CI',
    'Kenya': 'KE', 'Congo (Democratic Republic)': 'CD', 'Sierra Leone': 'SL',
    'Guinea': 'GN', 'Jordan': 'JO', 'Azerbaijan': 'AZ', 'Kosovo': 'XK',
    'Senegal': 'SN', 'Mauritania': 'MR', 'Lebanon': 'LB', 'Saudi Arabia': 'SA',
}

# ── EU27 Push: total detections by nationality × year ──
df_eu = pd.read_csv('data/eurostat_migr_eipre_eu27.csv')
eu_push = (
    df_eu[(df_eu['reason_code'] == 'TOTAL') & (df_eu['citizen_code'] != 'TOTAL')]
    .groupby(['citizen_code', 'time_code'])['value']
    .sum().reset_index()
)
eu_push.columns = ['iso2', 'year', 'eu27_detections']

# ── UK arrivals: aggregate small boats to nationality × year ──
uk_annual = (
    df_sb.groupby(['Nationality', 'Year'])['Number of detections']
    .sum().reset_index()
)
uk_annual.columns = ['Nationality', 'year', 'uk_arrivals']
uk_annual['iso2'] = uk_annual['Nationality'].map(NAT_TO_ISO)

# ── Join on iso2 × year ──
decomp = uk_annual.dropna(subset=['iso2']).merge(eu_push, on=['iso2', 'year'], how='inner')

# RouteShare = UK arrivals / (EU27 + UK) — UK is not in EU27 post-Brexit
decomp['total_europe'] = decomp['eu27_detections'] + decomp['uk_arrivals']
decomp['route_share'] = decomp['uk_arrivals'] / decomp['total_europe']

print(f"Decomposition panel: {len(decomp)} obs | "
      f"{decomp['Nationality'].nunique()} nationalities | "
      f"years {decomp['year'].min()}–{decomp['year'].max()}")
decomp[['Nationality', 'year', 'uk_arrivals', 'eu27_detections', 'route_share']].head(10)

Decomposition panel: 221 obs | 43 nationalities | years 2018–2024


,Nationality,year,uk_arrivals,eu27_detections,route_share
0,Afghanistan,2018,3,30320,0.000099
1,Afghanistan,2019,69,56200,0.001226
2,Afghanistan,2020,494,34125,0.014270
3,Afghanistan,2021,1437,60480,0.023208
4,Afghanistan,2022,9088,114495,0.073538
5,Afghanistan,2023,5511,112555,0.046677
6,Afghanistan,2024,6065,60040,0.091748
7,Albania,2018,16,31240,0.000512
8,Albania,2019,13,33540,0.000387
9,Albania,2020,54,30950,0.001742


In [30]:
# ── Summary table: average Push and RouteShare per nationality ────────────────

summary = decomp.groupby('Nationality').agg(
    uk_total=('uk_arrivals', 'sum'),
    eu27_total=('eu27_detections', 'sum'),
    mean_route_share=('route_share', 'mean'),
    latest_route_share=('route_share', 'last'),
    n_years=('year', 'count'),
).sort_values('uk_total', ascending=False)

summary['total_europe'] = summary['uk_total'] + summary['eu27_total']

# RouteShare trend: latest year minus earliest year (positive = UK share growing)
rs_trend = decomp.sort_values('year').groupby('Nationality').apply(
    lambda g: g['route_share'].iloc[-1] - g['route_share'].iloc[0]
    if len(g) > 1 else 0.0
)
summary['rs_trend'] = rs_trend

print("Top 20 nationalities — Push vs RouteShare decomposition:")
cols = ['uk_total', 'eu27_total', 'total_europe', 'mean_route_share', 'rs_trend']
summary[cols].head(20).style.format({
    'uk_total': '{:,.0f}',
    'eu27_total': '{:,.0f}',
    'total_europe': '{:,.0f}',
    'mean_route_share': '{:.1%}',
    'rs_trend': '{:+.1%}',
})

Top 20 nationalities — Push vs RouteShare decomposition:


/var/folders/55/92g1s_gx6bqf0mxxtwxlddt00000gn/T/ipykernel_10449/2589283170.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rs_trend = decomp.sort_values('year').groupby('Nationality').apply(


,uk_total,eu27_total,total_europe,mean_route_share,rs_trend
Nationality,,,,,
Iran,"25,780","95,590","121,370",20.4%,+23.7%
Afghanistan,"22,667","468,215","490,882",3.6%,+9.2%
Iraq,"17,521","187,205","204,726",8.8%,+12.4%
Albania,"15,118","231,345","246,463",4.6%,+2.0%
Syria,"13,216","797,605","810,821",1.4%,+3.2%
Eritrea,"11,552","57,855","69,407",17.4%,+21.8%
Sudan,"8,201","42,430","50,631",16.3%,+3.0%
Vietnam,"6,996","38,495","45,491",12.2%,+30.5%
Turkey,"6,210","243,985","250,195",1.6%,+3.2%


In [31]:
# ── Scatter: Push (EU27 total) vs RouteShare — with policy quadrants ─────────

top_n = summary.head(20).copy()

fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    top_n['eu27_total'],
    top_n['mean_route_share'] * 100,
    s=np.sqrt(top_n['uk_total']) * 5,
    c=top_n['rs_trend'] * 100,
    cmap='RdBu_r',
    edgecolors='black',
    linewidth=0.6,
    alpha=0.85,
    vmin=-15, vmax=15,
)

for nat, row in top_n.iterrows():
    ax.annotate(
        nat,
        (row['eu27_total'], row['mean_route_share'] * 100),
        fontsize=9, fontweight='bold',
        xytext=(6, 4), textcoords='offset points',
    )

# Quadrant lines at medians
med_push = top_n['eu27_total'].median()
med_rs = top_n['mean_route_share'].median() * 100
ax.axvline(med_push, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(med_rs, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)

# Quadrant labels
bbox = dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8, edgecolor='grey')
ax.text(0.02, 0.98, 'UK-SPECIFIC\nSmall EU flow but UK gets large share\n→ Bilateral / transit deal',
        transform=ax.transAxes, va='top', ha='left', fontsize=8, color='#555', bbox=bbox)
ax.text(0.98, 0.98, 'HIGHEST PRIORITY\nLarge EU flow AND UK-targeted\n→ Source + bilateral intervention',
        transform=ax.transAxes, va='top', ha='right', fontsize=8, color='#555', bbox=bbox)
ax.text(0.02, 0.02, 'LOW PRIORITY\nSmall flow, small UK share',
        transform=ax.transAxes, va='bottom', ha='left', fontsize=8, color='#555', bbox=bbox)
ax.text(0.98, 0.02, 'EUROPEAN PROBLEM\nLarge EU flow but UK share is small\n→ Multilateral / EU burden-sharing',
        transform=ax.transAxes, va='bottom', ha='right', fontsize=8, color='#555', bbox=bbox)

ax.set_xlabel('Total EU27 Detections (Push), 2018–2024', fontsize=11)
ax.set_ylabel('UK RouteShare (%)', fontsize=11)
ax.set_title(
    'Model 3 — Source–Transit Decomposition\n'
    'Bubble size = UK arrivals  |  Colour = RouteShare trend (red = UK share growing)',
    fontsize=13, fontweight='bold',
)
plt.colorbar(scatter, label='RouteShare trend (pp change)', shrink=0.7)
ax.set_xscale('log')

fig.tight_layout()
plt.savefig('presentation/model3_push_routeshare.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/55/92g1s_gx6bqf0mxxtwxlddt00000gn/T/ipykernel_10449/1833723813.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
# ── RouteShare over time for top nationalities ───────────────────────────────

top6 = summary.head(6).index.tolist()
rs_ts = decomp[decomp['Nationality'].isin(top6)].pivot(
    index='year', columns='Nationality', values='route_share'
) * 100

fig, ax = plt.subplots(figsize=(11, 5))
rs_ts.plot(ax=ax, marker='o', linewidth=2)
ax.set_ylabel('UK RouteShare (%)')
ax.set_xlabel('Year')
ax.set_title('UK Share of European Flow Over Time — Top 6 Nationalities', fontweight='bold')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_ylim(bottom=0)
fig.tight_layout()
plt.savefig('presentation/model3_routeshare_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/55/92g1s_gx6bqf0mxxtwxlddt00000gn/T/ipykernel_10449/2691832801.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
# ── Intervention-type recommendation per country ─────────────────────────────

def classify_intervention(row):
    high_push = row['eu27_total'] > med_push_raw
    high_rs = row['mean_route_share'] > med_rs_raw
    if high_push and high_rs:
        return 'Source + bilateral'
    elif high_push and not high_rs:
        return 'Multilateral / EU burden-sharing'
    elif not high_push and high_rs:
        return 'Bilateral / transit deal'
    else:
        return 'Low priority'

med_push_raw = summary.head(20)['eu27_total'].median()
med_rs_raw = summary.head(20)['mean_route_share'].median()

summary['intervention_type'] = summary.head(20).apply(classify_intervention, axis=1)

recs = summary.dropna(subset=['intervention_type']).sort_values('uk_total', ascending=False)
recs[['uk_total', 'eu27_total', 'mean_route_share', 'rs_trend', 'intervention_type']].style.format({
    'uk_total': '{:,.0f}',
    'eu27_total': '{:,.0f}',
    'mean_route_share': '{:.1%}',
    'rs_trend': '{:+.1%}',
})

,uk_total,eu27_total,mean_route_share,rs_trend,intervention_type
Nationality,,,,,
Iran,"25,780","95,590",20.4%,+23.7%,Source + bilateral
Afghanistan,"22,667","468,215",3.6%,+9.2%,Multilateral / EU burden-sharing
Iraq,"17,521","187,205",8.8%,+12.4%,Source + bilateral
Albania,"15,118","231,345",4.6%,+2.0%,Multilateral / EU burden-sharing
Syria,"13,216","797,605",1.4%,+3.2%,Multilateral / EU burden-sharing
Eritrea,"11,552","57,855",17.4%,+21.8%,Bilateral / transit deal
Sudan,"8,201","42,430",16.3%,+3.0%,Bilateral / transit deal
Vietnam,"6,996","38,495",12.2%,+30.5%,Bilateral / transit deal
Turkey,"6,210","243,985",1.6%,+3.2%,Multilateral / EU burden-sharing


## Event Impact Analysis — Do Interventions Work?

The models above tell us *where* to intervene and *how*. But do interventions actually reduce net flows? We can look at natural experiments — real-world policy shocks and geopolitical events — and examine whether the time series shows a structural break.

Three events provide testable cases:

| Event | Date | Mechanism |
|---|---|---|
| **UK-Albania communiqué** | 13 Dec 2022 | Bilateral returns deal + fast-track processing |
| **Israel strikes on Iran** | 26 Oct 2024 | Geopolitical shock in source country |
| **UK-France "one in, one out"** | 6 Aug 2025 | Transit-country returns agreement |

**Caveat**: this is *visual* evidence of structural breaks, not formal causal identification. Confounders include seasonality (Q1 is always low, Q3 always high), concurrent policy changes, and weather/crossing conditions. A proper diff-in-diff or synthetic control analysis would be needed to isolate the causal effect — but the time series is the essential first step.

In [34]:
# ── Build net flow panel (arrivals − returns) by nationality × quarter ────────

df_arrivals = pd.read_excel(
    'data/illegal-entry-routes-to-the-uk-dataset-dec-2025.xlsx',
    sheet_name='Data_IER_D01', header=1,
)
df_returns = pd.read_excel(
    'data/returns-datasets-dec-2025.xlsx',
    sheet_name='Data_Ret_D01', header=1,
)

df_sb = df_arrivals[df_arrivals['Method of entry'] == 'Small boat arrivals']

arr_q = (
    df_sb.groupby(['Nationality', 'Year', 'Quarter'])['Number of detections']
    .sum().reset_index()
)
arr_q.columns = ['Nationality', 'Year', 'Quarter', 'arrivals']

ret_q = (
    df_returns.groupby(['Nationality', 'Year', 'Quarter'])['Number of returns']
    .sum().reset_index()
)
ret_q.columns = ['Nationality', 'Year', 'Quarter', 'returns']

nf = arr_q.merge(ret_q, on=['Nationality', 'Year', 'Quarter'], how='left')
nf['returns'] = nf['returns'].fillna(0)
nf['net_flow'] = nf['arrivals'] - nf['returns']

# Numeric quarter index for plotting
quarter_order = sorted(nf['Quarter'].unique())
q_to_idx = {q: i for i, q in enumerate(quarter_order)}
nf['q_idx'] = nf['Quarter'].map(q_to_idx)

# Total across all nationalities
nf_total = nf.groupby(['Quarter', 'q_idx']).agg(
    arrivals=('arrivals', 'sum'), returns=('returns', 'sum')
).reset_index()
nf_total['net_flow'] = nf_total['arrivals'] - nf_total['returns']

print(f"Net flow panel: {len(nf):,} rows | {nf['Nationality'].nunique()} nationalities | "
      f"{len(quarter_order)} quarters ({quarter_order[0]} to {quarter_order[-1]})")

Net flow panel: 1,171 rows | 111 nationalities | 32 quarters (2018 Q1 to 2025 Q4)


In [35]:
# ── 3-panel event impact time series ─────────────────────────────────────────

# Event positions (quarter index)
EVENT_ALBANIA = q_to_idx['2022 Q4']
EVENT_IRAN    = q_to_idx['2024 Q4']
EVENT_FRANCE  = q_to_idx['2025 Q3']

alb = nf[nf['Nationality'] == 'Albania'].sort_values('q_idx')
irn = nf[nf['Nationality'] == 'Iran'].sort_values('q_idx')
tot = nf_total.sort_values('q_idx')

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# ── Tick labels: show every 4th quarter ──
tick_positions = list(range(0, len(quarter_order), 4))
tick_labels = [quarter_order[i] for i in tick_positions]

def plot_panel(ax, data, x_col, event_idx, event_label, event_date, title, color):
    # Pre/post shading
    ax.axvspan(event_idx - 0.5, data[x_col].max() + 0.5, alpha=0.07, color='red', zorder=0)

    # Arrivals and returns as stacked area context
    ax.fill_between(data[x_col], 0, data['arrivals'], alpha=0.15, color=color, label='Arrivals')
    ax.fill_between(data[x_col], 0, -data['returns'], alpha=0.15, color='#2ecc71', label='Returns (negative)')

    # Net flow line
    ax.plot(data[x_col], data['net_flow'], color=color, linewidth=2.5, marker='o',
            markersize=4, zorder=5, label='Net flow (arrivals − returns)')
    ax.axhline(0, color='black', linewidth=0.6)

    # Event line
    ax.axvline(event_idx, color='#c0392b', linewidth=2, linestyle='--', zorder=6)
    y_top = ax.get_ylim()[1]
    ax.annotate(f'{event_label}\n({event_date})',
                xy=(event_idx, y_top * 0.85), fontsize=8.5, fontweight='bold',
                color='#c0392b', ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor='#c0392b'))

    ax.set_title(title, fontsize=12, fontweight='bold', loc='left')
    ax.set_ylabel('People per quarter')
    ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

# Panel A: Albania
plot_panel(axes[0], alb, 'q_idx', EVENT_ALBANIA,
           'UK-Albania\ncommuniqué', 'Dec 2022',
           'A — Albania: bilateral returns deal collapsed arrivals', '#3498db')

# Panel B: Iran
plot_panel(axes[1], irn, 'q_idx', EVENT_IRAN,
           'Israel strikes\nIran', 'Oct 2024',
           'B — Iran: geopolitical shock — effect unclear (seasonal noise)', '#e67e22')

# Panel C: Total
plot_panel(axes[2], tot, 'q_idx', EVENT_FRANCE,
           'UK-France\n"one in, one out"', 'Aug 2025',
           'C — All nationalities: France deal — too early to assess', '#8e44ad')

axes[2].set_xlabel('Quarter')
axes[2].set_xticks(tick_positions)
axes[2].set_xticklabels(tick_labels, rotation=45, ha='right')

fig.suptitle('Event Impact on Net Migration Flow (Arrivals − Returns)',
             fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
plt.savefig('presentation/model5_event_impacts.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/55/92g1s_gx6bqf0mxxtwxlddt00000gn/T/ipykernel_10449/3221516425.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
# ── Before / after summary table ─────────────────────────────────────────────

WINDOW = 4  # quarters before and after

def before_after(data, x_col, event_idx, label):
    pre  = data[(data[x_col] >= event_idx - WINDOW) & (data[x_col] < event_idx)]
    post = data[(data[x_col] >= event_idx) & (data[x_col] < event_idx + WINDOW)]
    return {
        'Event': label,
        'Pre mean arrivals':  pre['arrivals'].mean(),
        'Post mean arrivals': post['arrivals'].mean(),
        'Δ arrivals':         post['arrivals'].mean() - pre['arrivals'].mean(),
        'Pre mean net flow':  pre['net_flow'].mean(),
        'Post mean net flow': post['net_flow'].mean(),
        'Δ net flow':         post['net_flow'].mean() - pre['net_flow'].mean(),
    }

ba = pd.DataFrame([
    before_after(alb, 'q_idx', EVENT_ALBANIA, 'Albania deal (Dec 2022)'),
    before_after(irn, 'q_idx', EVENT_IRAN,    'Israel strikes Iran (Oct 2024)'),
    before_after(tot, 'q_idx', EVENT_FRANCE,  'France "1-in-1-out" (Aug 2025)'),
])
ba = ba.set_index('Event')

ba.style.format('{:,.0f}').bar(
    subset=['Δ arrivals', 'Δ net flow'],
    color=['#27ae60', '#e74c3c'],
    align='zero',
)

,Pre mean arrivals,Post mean arrivals,Δ arrivals,Pre mean net flow,Post mean net flow,Δ net flow
Event,,,,,,
Albania deal (Dec 2022),"3,006",512,"-2,494","2,420",-806,"-3,226"
Israel strikes Iran (Oct 2024),984,"1,195",211,914,"1,115",201
"France ""1-in-1-out"" (Aug 2025)","10,827","10,745",-82,"4,124","2,950","-1,175"


## Granger Causality: Are Countries' Migration Flows Linked?

**Granger causality** tests whether lagged arrivals from country A improve the prediction of arrivals from country B, beyond what B's own history can explain. This reveals whether migration flows are **linked across nationalities** — through shared transit routes, common push factors, or substitution effects.

We build an **N × N matrix** of Granger causality p-values across the top source countries. A significant cell (A → B) means "a change in arrivals from A *precedes* a change in arrivals from B."

**Policy insight**: if countries cluster into causally linked groups, intervening in one source country may have spillover effects on flows from others. Conversely, if two countries' flows are independent, they require separate interventions.

In [37]:
# ── Granger causality matrix: country × country ─────────────────────────────

from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

# Build quarterly arrivals per nationality (balanced panel, fill missing with 0)
df_sb_gc = df_arrivals[df_arrivals['Method of entry'] == 'Small boat arrivals']
arr_gc = df_sb_gc.groupby(['Nationality', 'Quarter'])['Number of detections'].sum().reset_index()
arr_gc.columns = ['Nationality', 'Quarter', 'arrivals']

quarter_order_gc = sorted(arr_gc['Quarter'].unique())

exclude = {'Not currently recorded', 'Stateless', 'Refugee', 'Other and unknown'}
top_nats_gc = (
    arr_gc.groupby('Nationality')['arrivals'].sum()
    .sort_values(ascending=False)
    .drop(labels=exclude, errors='ignore')
    .head(10).index.tolist()
)

# Pivot to wide format: rows = quarters, columns = nationalities
wide = arr_gc[arr_gc['Nationality'].isin(top_nats_gc)].pivot(
    index='Quarter', columns='Nationality', values='arrivals'
).reindex(quarter_order_gc).fillna(0)

MAX_LAG = 4

# N × N Granger causality p-value matrix
# Cell (row=cause, col=effect): does row Granger-cause col?
gc_pvals = pd.DataFrame(np.nan, index=top_nats_gc, columns=top_nats_gc)

for cause in top_nats_gc:
    for effect in top_nats_gc:
        if cause == effect:
            continue
        data = wide[[effect, cause]].values  # grangercausalitytests: col 0 = Y, col 1 = X
        try:
            gc = grangercausalitytests(data, maxlag=MAX_LAG, verbose=False)
            best_lag = min(gc.keys(), key=lambda k: gc[k][0]['ssr_ftest'][1])
            gc_pvals.loc[cause, effect] = gc[best_lag][0]['ssr_ftest'][1]
        except:
            pass

print(f"Granger causality matrix: {len(top_nats_gc)} × {len(top_nats_gc)} countries")
print(f"Cell (row → col): p-value that row Granger-causes col")
print(f"Significant pairs (p < 0.05):")
for cause in top_nats_gc:
    for effect in top_nats_gc:
        p = gc_pvals.loc[cause, effect]
        if not np.isnan(p) and p < 0.05:
            print(f"  {cause:15s} → {effect:15s}  p = {p:.4f}")

Granger causality matrix: 10 × 10 countries
Cell (row → col): p-value that row Granger-causes col
Significant pairs (p < 0.05):
  Iran            → Afghanistan      p = 0.0000
  Iran            → Iraq             p = 0.0297
  Iran            → Albania          p = 0.0000
  Iran            → Turkey           p = 0.0173
  Afghanistan     → Vietnam          p = 0.0018
  Afghanistan     → Turkey           p = 0.0000
  Iraq            → Iran             p = 0.0297
  Iraq            → Afghanistan      p = 0.0000
  Iraq            → Albania          p = 0.0000
  Iraq            → Turkey           p = 0.0183
  Eritrea         → Iran             p = 0.0058
  Eritrea         → Afghanistan      p = 0.0282
  Eritrea         → Iraq             p = 0.0052
  Eritrea         → Albania          p = 0.0044
  Eritrea         → Vietnam          p = 0.0136
  Eritrea         → Somalia          p = 0.0191
  Albania         → Afghanistan      p = 0.0000
  Albania         → Turkey           p = 0.0000
  Syria 

In [39]:
# ── Heatmap: Granger causality matrix ────────────────────────────────────────

import matplotlib.colors as mcolors

# Transform to -log10(p) for visual scale — higher = more significant
gc_sig = -np.log10(gc_pvals.clip(lower=1e-10).astype(float))
np.fill_diagonal(gc_sig.values, 0)

fig, ax = plt.subplots(figsize=(10, 8))

# Custom colormap: white (not significant) → orange → red (highly significant)
cmap = mcolors.LinearSegmentedColormap.from_list('sig', ['#f7f7f7', '#fee8c8', '#e34a33'])
threshold_05 = -np.log10(0.05)

im = ax.imshow(gc_sig.values, cmap=cmap, vmin=0, vmax=5, aspect='equal')

# Mark significant cells with stars
for i in range(len(top_nats_gc)):
    for j in range(len(top_nats_gc)):
        if i == j:
            ax.text(j, i, '—', ha='center', va='center', fontsize=9, color='#ccc')
            continue
        p = gc_pvals.iloc[i, j]
        if np.isnan(p):
            continue
        label = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
        color = 'white' if p < 0.01 else 'black'
        ax.text(j, i, f'{label}', ha='center', va='center', fontsize=10, fontweight='bold', color=color)

ax.set_xticks(range(len(top_nats_gc)))
ax.set_xticklabels(top_nats_gc, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(top_nats_gc)))
ax.set_yticklabels(top_nats_gc, fontsize=9)
ax.set_xlabel('Effect (arrivals predicted)', fontsize=11)
ax.set_ylabel('Cause (lagged arrivals)', fontsize=11)
ax.set_title(
    'Granger Causality Matrix — Country × Country\n'
    'Cell (row → col): does a change in row arrivals predict a change in col arrivals?\n'
    '*** p<0.01  ** p<0.05  * p<0.1',
    fontsize=12, fontweight='bold',
)
plt.colorbar(im, label='−log₁₀(p-value)', shrink=0.8)

fig.tight_layout()
plt.savefig('presentation/granger_matrix.png', dpi=150, bbox_inches='tight')
plt.show()